# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [2]:
# Write your code below.

%load_ext dotenv
%dotenv 

In [1]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [14]:
import os
from glob import glob

# Write your code below.
parquet_files = glob(os.path.join(os.getenv('PRICE_DATA'), "**/*.parquet"), recursive= True)
dd_px = dd.read_parquet(parquet_files).set_index("ticker")


In [17]:
dd_px.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'Year'],
      dtype='object')

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [20]:
# Write your code below.

dd_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1),
                       Adj_Close_lag_1 = x['Adj Close'].shift(1))
)

dd_rets = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)

dd_range = dd_rets.assign(
    hi_lo_range = lambda x : x['High'] - x['Low']
)

C:\Users\user\AppData\Local\Temp\ipykernel_14036\549486418.py:3: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  dd_shift = dd_px.groupby('ticker', group_keys=False).apply(


In [21]:
dd_rets

ImportError: Dask diagnostics requirements are not installed.

Please either conda or pip install as follows:

  conda install dask                     # either conda install
  python -m pip install "dask[diagnostics]" --upgrade  # or python -m pip install

Dask DataFrame Structure:
                          Date     Open     High      Low    Close Adj Close   Volume  source   Year Close_lag_1 Adj_Close_lag_1  Returns
npartitions=89                                                                                                                           
AA              datetime64[ns]  float64  float64  float64  float64   float64  float64  string  int32     float64         float64  float64
ADX                        ...      ...      ...      ...      ...       ...      ...     ...    ...         ...             ...      ...
...                        ...      ...      ...      ...      ...       ...      ...     ...    ...         ...             ...      ...
WGO                        ...      ...      ...      ...      ...       ...      ...     ...    ...         ...             ...      ...
WGO                        ...      ...      ...      ...      ...       ...      ...     ...    ...         ...             ...      ...
Dask Nam

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [23]:
# Write your code below.

dd_rets.compute()

dd_moving_avg = (dd_rets.groupby('ticker', group_keys= False)
                 .apply(
                     lambda x : x.sort_values('Date')
                     .assign(
                         mov_avg_10_days = x['Returns'].rolling(10).mean()
                     )
                 )

)

C:\Users\user\AppData\Local\Temp\ipykernel_14036\2241694535.py:6: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  .apply(


In [25]:
dd_moving_avg.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,mov_avg_10_days
ticker,,,,,,,,,,,,,
AA,1962-01-02,6.532155,6.556185,6.532155,6.532155,1.536658,55900.0,AA.csv,1962,36.982170,34.542450,-0.823370,NaN
AA,1962-01-03,6.532155,6.632280,6.524145,6.632280,1.560212,74500.0,AA.csv,1962,6.532155,1.536658,0.015328,NaN
AA,1962-01-04,6.632280,6.664320,6.632280,6.632280,1.560212,80500.0,AA.csv,1962,6.632280,1.560212,0.000000,NaN
AA,1962-01-05,6.632280,6.656310,6.616260,6.624270,1.558326,70500.0,AA.csv,1962,6.632280,1.560212,-0.001208,NaN
AA,1962-01-08,6.608250,6.608250,6.339915,6.408000,1.507450,93800.0,AA.csv,1962,6.624270,1.558326,-0.032648,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
WGO,2020-03-26,29.580000,34.099998,29.580000,31.070000,31.070000,1511100.0,WGO.csv,2020,29.190001,29.190001,0.064406,0.006128
WGO,2020-03-27,29.190001,30.830000,28.620001,30.350000,30.350000,867200.0,WGO.csv,2020,31.070000,31.070000,-0.023173,0.009116
WGO,2020-03-30,29.719999,30.840000,29.100000,29.780001,29.780001,1111400.0,WGO.csv,2020,30.350000,30.350000,-0.018781,0.003134


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

For smaller datasets and complex functions like moving mean, it is faster to perform in pandas compared to dask dataframe. Whereas if we have large datasets then it would be efficient to perform functions in dask dataframes.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.